# Museum Visitor vs City Population Analysis

In [ ]:
import os
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

API_URL = os.getenv("API_URL", "http://localhost:8000")

# Trigger ingestion (idempotent)
resp = requests.post(f"{API_URL}/ingest")
resp.raise_for_status()
print(resp.json())

In [ ]:
museums_df = pd.DataFrame(requests.get(f"{API_URL}/museums").json())
cities_df = pd.DataFrame(requests.get(f"{API_URL}/cities").json())
print(f"{len(museums_df)} museums across {len(cities_df)} cities")
museums_df.head(10)

In [ ]:
reg = requests.get(f"{API_URL}/regression").json()
print(f"R\u00b2 = {reg['r_squared']:.3f}")
print(f"slope = {reg['slope']:.2f}  (visitors per capita)")
print(f"intercept = {reg['intercept']:,.0f}")

In [ ]:
preds_df = pd.DataFrame(reg["predictions"])

fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=preds_df, x="population", y="visitors", ax=ax, s=80)

for _, row in preds_df.iterrows():
    ax.annotate(row["city"], (row["population"], row["visitors"]), fontsize=7, alpha=0.7)

x_range = np.linspace(preds_df["population"].min(), preds_df["population"].max(), 100)
y_range = reg["slope"] * x_range + reg["intercept"]
ax.plot(x_range, y_range, color="red", linewidth=2, label=f"R\u00b2={reg['r_squared']:.2f}")

ax.set_xlabel("City Population")
ax.set_ylabel("Annual Museum Visitors")
ax.set_title("Museum Visitors vs City Population")
ax.legend()
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
plt.tight_layout()
plt.show()

## Interpretation

The **R²** value measures how much of the variance in annual museum visitors is explained by city population alone. An R² close to 1.0 indicates a strong linear relationship — larger cities reliably attract more museum visitors — while a low R² (below ~0.5) suggests that population is only a weak predictor and other factors (tourism infrastructure, number of museums, cultural funding) play a significant role.

**Outliers** to watch for:
- Cities that sit well **above** the regression line are over-performing relative to their population — likely major tourist destinations or cities with world-class museum collections.
- Cities that sit well **below** the line are under-performing — potentially due to fewer institutions, lower tourism draw, or data gaps in the scraped source.

The slope (visitors per capita) gives a baseline expectation: for every additional resident a city has, visitor counts are expected to rise by that amount on average.